# M4c — forced-choice label-free prompt

Reproduces the headline tables from `docs/insights/m4c_fc_label_free.md`.
Tests whether M4b's `ball ≈ no-label, circle = suppressor` H2 finding
survives a switch from open prompt to forced-choice MCQ on Qwen, and
whether the LLaVA "A" bias from M6 r1 relaxes when the FC options use
`The depicted object` instead of `It`/`{label}`.

Depends on:
- `outputs/fc_label_free_qwen_20260425-042817_eec92f1a/`
- `outputs/fc_label_free_llava_20260425-044517_81ae56d5/`
- `outputs/label_free_20260425-031430_315c5318/` (M4b open-prompt baseline)
- `outputs/mvp_full_20260424-094103_8ae1fa3d/` (M2 forced-choice labeled)

If the FC label-free runs don't exist yet:
```bash
uv run python scripts/02_run_inference.py --config configs/fc_label_free_qwen.py  --stimulus-dir inputs/mvp_full_20260424-093926_e9d79da3
uv run python scripts/02_run_inference.py --config configs/fc_label_free_llava.py --stimulus-dir inputs/mvp_full_20260424-093926_e9d79da3
uv run python scripts/03_score_and_summarize.py --run-dir outputs/fc_label_free_<model>_<ts>
```

In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from physical_mode.metrics.first_letter import extract_first_letter

QFC = pd.read_parquet(PROJECT_ROOT / 'outputs' / 'fc_label_free_qwen_20260425-042817_eec92f1a'  / 'predictions_scored.parquet')
LFC = pd.read_parquet(PROJECT_ROOT / 'outputs' / 'fc_label_free_llava_20260425-044517_81ae56d5' / 'predictions_scored.parquet')
QO  = pd.read_parquet(PROJECT_ROOT / 'outputs' / 'label_free_20260425-031430_315c5318'         / 'predictions_scored.parquet')
M2  = pd.read_parquet(PROJECT_ROOT / 'outputs' / 'mvp_full_20260424-094103_8ae1fa3d'           / 'predictions_scored.parquet')
M2_FC = M2[M2.prompt_variant == 'forced_choice']

QFC['first_letter'] = QFC.raw_text.apply(extract_first_letter)
LFC['first_letter'] = LFC.raw_text.apply(extract_first_letter)
print(f'Qwen FC label-free  n={len(QFC)}')
print(f'LLaVA FC label-free n={len(LFC)}')
print(f'Qwen open label-free n={len(QO)}')
print(f'M2 FC labeled       n={len(M2_FC)}')

Qwen FC label-free  n=480
LLaVA FC label-free n=480
Qwen open label-free n=480
M2 FC labeled       n=1440


## Qwen — first_letter distribution under FC label-free

In [2]:
QFC.first_letter.value_counts().to_frame('count')

,count
first_letter,
A,292
B,77
D,57
other,54


## Headline 1 — paired delta vs M2 FC labeled (Qwen)

If M4b's H2 reframing survives the prompt-format shift to FC, we
expect `ball − _nolabel ≈ 0` and `circle − _nolabel < 0`. Planet's
delta will tell us whether FC's gravity-centric option set masks
orbital regime.

In [3]:
fc_nl = QFC.groupby('sample_id').pmr.mean().rename('_nolabel')
rows = []
for lab in ['ball', 'circle', 'planet']:
    fc_lab = M2_FC[M2_FC.label == lab].groupby('sample_id').pmr.mean().rename(lab)
    j = pd.concat([fc_nl, fc_lab], axis=1).dropna()
    rows.append({'label': lab, 'mean_delta': float((j[lab] - j._nolabel).mean()), 'n': len(j)})
pd.DataFrame(rows).set_index('label').round(3)

,mean_delta,n
label,,
ball,0.012,480
circle,-0.208,480
planet,-0.262,480


Reading: `ball − _nolabel ≈ 0` (matches M4b open). `circle − _nolabel`
is much more negative than M4b (−0.21 vs −0.07). `planet − _nolabel`
is newly negative — orbital responses collapse to FC's D option
because A/B/C are all gravity-centric.

## Headline 2 — paired open-vs-FC delta on Qwen no-label

How much more conservative is FC than open at the same image,
without label confounding?

In [4]:
fc_pmr = QFC.groupby('sample_id').pmr.mean().rename('fc')
op_pmr = QO.groupby('sample_id').pmr.mean().rename('open')
j = pd.concat([fc_pmr, op_pmr], axis=1).dropna()
delta = j.fc - j.open
print(f'PMR(FC, _nolabel) − PMR(open, _nolabel)  mean = {delta.mean():+.3f}, n = {len(j)}')

PMR(FC, _nolabel) − PMR(open, _nolabel)  mean = -0.131, n = 480


## Cell-level — `line/blank/none` (the FC abstract sink)

Under FC, every label condition collapses to D=10/10 (or 9/10 for
no-label) at the fully abstract image. The label is no longer the
discriminating axis — FC's D option pulls everything in.

In [5]:
cell_filter = lambda d: d[(d.object_level == 'line') & (d.bg_level == 'blank') & (d.cue_level == 'none')]
rows = []
rows.append(('open × _nolabel (M4b)', cell_filter(QO)))
rows.append(('FC × _nolabel  (M4c)', cell_filter(QFC)))
for lab in ['ball', 'circle', 'planet']:
    rows.append((f'FC × {lab}', cell_filter(M2_FC[M2_FC.label == lab])))
out = pd.DataFrame([
    {'condition': name, 'pmr': sub.pmr.mean(), 'hold_still': sub.hold_still.mean(),
     'abstract_reject': sub.abstract_reject.mean(), 'n': len(sub)}
    for name, sub in rows
]).set_index('condition')
out.round(2)

,pmr,hold_still,abstract_reject,n
condition,,,,
open × _nolabel (M4b),0.4,0.2,0.0,10
FC × _nolabel (M4c),0.0,0.4,1.0,10
FC × ball,0.0,0.6,1.0,10
FC × circle,0.0,1.0,1.0,10
FC × planet,0.0,0.1,1.0,10


## LLaVA — does re-template fix the "A" bias?

M6 r1 found 12/12 `A` responses on a labeled FC smoke. Re-templating
the options to use `The depicted object` (this run) tests whether
the bias was driven by the antecedent.

In [6]:
LFC.first_letter.value_counts().to_frame('count')

,count
first_letter,
A,477
B,3


477/480 = 99.4 % `A`. Re-templating does **not** relax the bias —
it's a model-level pathology, not a prompt artefact. FC is unusable
on LLaVA-1.5 with verb-PMR or first-letter metrics. Round 2 idea:
first-token logit ratio scoring.

In [7]:
# Quick check — does any cell pull LLaVA off A?
LFC.groupby(['object_level', 'first_letter']).size().unstack(fill_value=0)

first_letter,A,B
object_level,,
filled,119,1
line,118,2
shaded,120,0
textured,120,0
